## Vloženia

V našom predchádzajúcom príklade sme pracovali s vysokodimenzionálnymi vektormi typu bag-of-words s dĺžkou `vocab_size` a explicitne sme prevádzali z nízkodimenzionálnych vektorov pozičnej reprezentácie do riedkej one-hot reprezentácie. Táto one-hot reprezentácia nie je pamäťovo efektívna, okrem toho, každé slovo je považované nezávisle od ostatných, t.j. one-hot kódované vektory nevyjadrujú žiadne sémantické podobnosti medzi slovami.

V tejto časti budeme pokračovať v skúmaní datasetu **News AG**. Na začiatok načítajme dáta a získajme niektoré definície z predchádzajúceho zápisníka.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Čo je embedding?

Myšlienka **embeddingu** je reprezentovať slová nižšerozmernými hustými vektormi, ktoré nejako odrážajú sémantický význam slova. Neskôr sa budeme zaoberať tým, ako vytvárať zmysluplné word embeddingy, ale zatiaľ si predstavme embeddingy ako spôsob zníženia dimenzionality vektorov slov.

Takže embeddingová vrstva by brala slovo ako vstup a produkovala by výstupný vektor špecifikovanej veľkosti `embedding_size`. V istom zmysle je veľmi podobná vrstve `Linear`, ale namiesto toho, aby brala one-hot zakódovaný vektor, bude schopná brať číslo slova ako vstup.

Použitím embeddingovej vrstvy ako prvej vrstvy v našej sieti môžeme prejsť z modelu bag-of-words na model **embedding bag**, kde najskôr prevedieme každé slovo v našom texte na príslušný embedding a potom vypočítame nejakú agregačnú funkciu nad všetkými týmito embeddingmi, ako napríklad `sum`, `average` alebo `max`.

![Obrázok zobrazujúci embeddingový klasifikátor pre päť slov v sekvencii.](../../../../../translated_images/sk/embedding-classifier-example.b77f021a7ee67eee.webp)

Naša klasifikačná neurónová sieť začne embeddingovou vrstvou, potom bude nasledovať agregačná vrstva a na vrchu lineárny klasifikátor:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Riešenie premennej veľkosti sekvencie

V dôsledku tejto architektúry je potrebné vytvárať minibatches pre našu sieť určitým spôsobom. V predchádzajúcej jednotke, pri použití bag-of-words, mali všetky BoW tenzory v minibatch rovnako veľkú veľkosť `vocab_size`, bez ohľadu na skutočnú dĺžku našej textovej sekvencie. Keď sa presunieme k word embeddings, skončíme s premenlivým počtom slov v každom textovom vzorke, a pri kombinovaní týchto vzoriek do minibatch budeme musieť použiť nejaký padding.

Toto je možné uskutočniť použitím rovnakého postupu poskytnutia funkcie `collate_fn` do datasource:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Tréning klasifikátora vkladania

Teraz, keď sme definovali správny dátový načítač, môžeme trénovať model pomocou tréningovej funkcie, ktorú sme definovali v predchádzajúcej časti:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Poznámka**: Tu trénujeme iba na 25 000 záznamoch (menej ako jednu plnú epóchu) kvôli času, ale môžete pokračovať v trénovaní, napísať funkciu pre trénovanie niekoľkých epôch a experimentovať s parametrom rýchlosti učenia, aby ste dosiahli vyššiu presnosť. Mali by ste byť schopní dosiahnuť presnosť okolo 90%.


### Vrstva EmbeddingBag a reprezentácia sekvencií s premenlivou dĺžkou

V predchádzajúcej architektúre sme museli všetky sekvencie doplniť na rovnakú dĺžku, aby sme ich mohli vložiť do minibatchu. Toto nie je najefektívnejší spôsob reprezentácie sekvencií s premennou dĺžkou - ďalším prístupom by bolo použiť vektor **offsetov**, ktorý by obsahoval offsety všetkých sekvencií uložených v jednom veľkom vektore.

![Obrázok zobrazujúci reprezentáciu sekvencie pomocou offsetov](../../../../../translated_images/sk/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Poznámka**: Na obrázku vyššie ukazujeme sekvenciu znakov, ale v našom príklade pracujeme so sekvenciami slov. Všeobecný princíp reprezentácie sekvencií pomocou vektora offsetov však zostáva rovnaký.

Na prácu s offsetovou reprezentáciou používame vrstvu [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Je podobná vrstve `Embedding`, ale ako vstup berie obsahový vektor a offsetový vektor a zároveň obsahuje priemernú vrstvu, ktorá môže byť `mean`, `sum` alebo `max`.

Tu je modifikovaná sieť, ktorá používa `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Na prípravu datasetu na trénovanie potrebujeme poskytnúť konverznú funkciu, ktorá pripraví vektor posunu:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Poznámka, že na rozdiel od všetkých predchádzajúcich príkladov naša sieť teraz prijíma dva parametre: dátový vektor a posuvný vektor, ktoré majú rôzne veľkosti. Podobne náš dátový načítač nám tiež poskytuje 3 hodnoty namiesto 2: oba textový a posuvný vektor sú poskytované ako vlastnosti. Preto potrebujeme mierne upraviť našu trénovaciu funkciu, aby sa o to postarala:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Sémantické vektorové reprezentácie: Word2Vec

V našom predchádzajúcom príklade sa učila vrstve vkladania modelu mapovať slová na vektorovú reprezentáciu, avšak táto reprezentácia nemala veľký sémantický význam. Bolo by pekné naučiť sa takú vektorovú reprezentáciu, že podobné slová alebo synonyma by zodpovedali vektorom, ktoré sú navzájom blízke podľa niektorej vektorovej vzdialenosti (napr. euklidovskej vzdialenosti).

Aby sme to dosiahli, potrebujeme náš model vkladania predtrénovať na veľkej zbierke textu špecifickým spôsobom. Jedným z prvých spôsobov trénovania sémantických vektorov je metóda nazývaná [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Je založená na dvoch hlavných architektúrach, ktoré sa používajú na vytvorenie distribuovanej reprezentácie slov:

 - **Continuous bag-of-words** (CBoW) — v tejto architektúre trénujeme model predpovedať slovo podľa okolitého kontextu. Pri n-gramu $(W_{-2},W_{-1},W_0,W_1,W_2)$ je cieľom modelu predpovedať $W_0$ zo slov $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** je opačný k CBoW. Model používa okolitý okno kontextových slov na predpovedanie súčasného slova.

CBoW je rýchlejšie, zatiaľ čo skip-gram je pomalší, ale lepšie reprezentuje zriedkavé slová.

![Obrázok zobrazujúci algoritmy CBoW a Skip-Gram na prevod slov na vektory.](../../../../../translated_images/sk/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Na experimentovanie s word2vec vektorom predtrénovaným na dátovej sade Google News môžeme použiť knižnicu **gensim**. Nižšie nájdeme slová najpodobnejšie k slovu 'neural'

> **Poznámka:** Keď prvýkrát vytvárate vektory slov, ich stiahnutie môže chvíľu trvať!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Môžeme tiež vypočítať vektorové vloženia z daného slova, ktoré sa použijú pri trénovaní klasifikačného modelu (pre prehľadnosť zobrazujeme iba prvých 20 komponentov vektora):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Skvelé na semantických vloženiach je, že môžete manipulovať s vektorovým kódovaním na zmenu sémantiky. Napríklad môžeme požiadať o nájdenie slova, ktorého vektorová reprezentácia by bola čo najbližšie slovám *king* a *woman*, a čo najďalej od slova *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Obe metódy CBoW aj Skip-Grams sú „prediktívne“ vektory, pretože berú do úvahy iba lokálny kontext. Word2Vec nevyužíva globálny kontext.

**FastText** stavia na Word2Vec učením vektorových reprezentácií pre každé slovo a znakovými n-grammi, ktoré sa vyskytujú v každom slove. Hodnoty reprezentácií sa potom pri každom kroku trénovania spriemerujú do jedného vektora. Hoci to pridáva veľa ďalších výpočtov predtréningu, umožňuje to vektorom slov zakódovať informácie na sub-slovnej úrovni.

Ďalšia metóda, **GloVe**, využíva myšlienku matice spoločného výskytu, používa neurónové metódy na dekompozíciu matice spoločného výskytu do expresívnejších a nelineárnych vektorov slov.

Môžete si zahrať s príkladom zmenou vektorov na FastText a GloVe, keďže gensim podporuje niekoľko rôznych modelov vektorových reprezentácií slov.


## Použitie predtrénovaných vektorových reprezentácií v PyTorch

Môžeme upraviť vyššie uvedený príklad tak, aby sme prednaplnili maticu v našej embedding vrstve sémantickými vektormi, ako napríklad Word2Vec. Musíme zohľadniť, že slovníky predtrénovaných vektorov a náš textový korpus pravdepodobne nebudú zhodné, takže inicializujeme váhy pre chýbajúce slová náhodnými hodnotami:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Teraz si natrénujme náš model. Všimnite si, že čas potrebný na trénovanie modelu je výrazne dlhší ako v predchádzajúcom príklade, kvôli väčšej vrstve vnorenia a teda oveľa vyššiemu počtu parametrov. Tiež kvôli tomu môžeme potrebovať trénovať náš model na väčšom počte príkladov, ak chceme predísť pretrénovaniu.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

V našom prípade nevidíme výrazný nárast presnosti, čo je pravdepodobne spôsobené veľmi odlišnými slovníkmi.
Na preklenutie problému rôznych slovníkov môžeme použiť jedno z nasledujúcich riešení:
* Pretrénovať model word2vec na našom slovníku
* Načítať našu dátovú sadu so slovníkom z predtrénovaného modelu word2vec. Slovník použitý na načítanie dátovej sady je možné určiť počas načítania.

Druhý prístup sa zdá byť jednoduchší, najmä preto, že framework PyTorch `torchtext` obsahuje vstavanú podporu pre embeddingy. Môžeme napríklad inštancovať slovník založený na GloVe nasledovne:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Načítaná slovná zásoba má nasledujúce základné operácie:
* `vocab.stoi` slovník nám umožňuje konvertovať slovo na jeho index v slovníku
* `vocab.itos` robí opačne - konvertuje číslo na slovo
* `vocab.vectors` je pole vektorov embeddingu, takže na získanie embeddingu slova `s` musíme použiť `vocab.vectors[vocab.stoi[s]]`

Tu je príklad manipulácie embeddingmi na demonštráciu rovnice **kind-man+woman = queen** (musel som mierne upraviť koeficient, aby to fungovalo):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Na natrénovanie klasifikátora pomocou týchto vektorov musíme najskôr zakódovať našu datovú množinu pomocou slovníka GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Ako sme videli vyššie, všetky vektorové vloženia sú uložené v matici `vocab.vectors`. Uľahčuje to načítanie týchto váh do váh vrstvy vloženia jednoduchým kopírovaním:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Teraz si natrénujeme náš model a uvidíme, či dosiahneme lepšie výsledky:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Jedným z dôvodov, prečo nevidíme výrazné zvýšenie presnosti, je fakt, že niektoré slová z nášho datasetu chýbajú v predtrénovanom slovníku GloVe, a preto sú v podstate ignorované. Aby sme tento fakt prekonali, môžeme trénovať vlastné vektorové reprezentácie na našom datasete. 


## Kontextové vkladania

Jedným kľúčovým obmedzením tradičných vopred trénovaných reprezentácií vkladania, ako je Word2Vec, je problém rozlíšenia významu slov. Hoci vopred trénované vkladania môžu zachytiť určitý význam slov v kontexte, každý možný význam slova je zakódovaný do rovnakého vkladania. To môže spôsobiť problémy v nasledujúcich modeloch, pretože mnohé slová ako slovo „play“ majú rôzne významy v závislosti od kontextu, v ktorom sú použité.

Napríklad slovo „play“ v týchto dvoch rôznych vetách má úplne odlišný význam:
- Išiel som na **hru** do divadla.
- John chce **hrať** sa so svojimi priateľmi.

Vopred trénované vkladania vyššie reprezentujú oba tieto významy slova „play“ v rovnakom vkladaní. Na prekonanie tohto obmedzenia potrebujeme vytvárať vkladania založené na **jazykovom modeli**, ktorý je trénovaný na veľkom korpuse textu a *vie*, ako je možné slová skladať do rôznych kontextov. Diskusia o kontextových vkladaní je mimo rozsahu tohto tutoriálu, ale vrátime sa k nim pri rozprávaní o jazykových modeloch v nasledujúcej jednotke.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
